In [3]:
from tensorflow import keras
import os
from PIL import Image

In [4]:
(train_images, train_labels), (test_images, test_labels) = keras.datasets.cifar10.load_data()

In [5]:
divisions = ['train','validation','others']
categories = ['airplane','ship','truck']

In [6]:
if not os.path.exists('./cifar10'):
	for division in divisions:
		for category in categories:
			os.makedirs(f'./cifar10/{division}/{category}')

In [7]:
def make_image_folder(label, category_name, ntrain, nvalidation):
	print(f'Making folder: label={label}, category_name={category_name},ntrain={ntrain},nvalidation={nvalidation}')
	count=0
	for i in range(train_images.shape[0]):
		if train_labels[i,0]==label:
			count+=1
			if count<=ntrain:
				division='train'
			elif count<=ntrain+nvalidation:
				division='validation'
			else:
				division='others'
			img = Image.fromarray(train_images[i])
			img.save(f'./cifar10/{division}/{category_name}/{i}.png')

In [8]:
make_image_folder(0,'airplane',800,200)
make_image_folder(8,'ship',800,200)
make_image_folder(9,'truck',800,200)

Making folder: label=0, category_name=airplane,ntrain=800,nvalidation=200


Making folder: label=8, category_name=ship,ntrain=800,nvalidation=200
Making folder: label=9, category_name=truck,ntrain=800,nvalidation=200


In [9]:
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.utils import image_dataset_from_directory

In [10]:
categories = ['airplane','ship','truck']
ncategories = len(categories)

In [11]:
inputs=keras.Input(shape=(32,32,3))
x=layers.Rescaling(1./255)(inputs)
x=layers.Conv2D(filters=32,kernel_size=3,activation='relu')(x)
x=layers.Conv2D(filters=32,kernel_size=3,activation='relu')(x)
x=layers.MaxPooling2D(pool_size=2)(x)
x=layers.Dropout(0.25)(x)
x=layers.Conv2D(filters=32,kernel_size=3,activation='relu')(x)
x=layers.Conv2D(filters=32,kernel_size=3,activation='relu')(x)
x=layers.MaxPooling2D(pool_size=2)(x)
x=layers.Dropout(0.25)(x)
x=layers.Flatten()(x)
x=layers.Dense(512,activation='relu')(x)
x=layers.Dropout(0.5)(x)
outputs=layers.Dense(units=ncategories,activation='softmax')(x)
model=keras.Model(inputs=inputs,outputs=outputs)

E0000 00:00:1779251446.300669   11497 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [12]:
model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

In [13]:
train_dataset = image_dataset_from_directory('./cifar10/train', image_size=(32,32), batch_size=32)
validation_dataset = image_dataset_from_directory('./cifar10/validation', image_size=(32,32), batch_size=32)
callbacks = [keras.callbacks.ModelCheckpoint(filepath='./dsx06_2.keras', save_best_only=True, monitor="val_loss")]

Found 2400 files belonging to 3 classes.
Found 600 files belonging to 3 classes.


In [14]:
history = model.fit(
    train_dataset,epochs=30,validation_data=validation_dataset,callbacks=callbacks
)

Epoch 1/30
 1/75 ━━━━━━━━━━━━━━━━━━━━ 2:16 2s/step - accuracy: 0.2812 - loss: 1.1178

W0000 00:00:1779251519.538249   11731 cpu_allocator_impl.cc:82] Allocation of 22003200 exceeds 10% of free system memory.
W0000 00:00:1779251519.539994   11738 cpu_allocator_impl.cc:82] Allocation of 22003200 exceeds 10% of free system memory.
W0000 00:00:1779251519.543727   11738 cpu_allocator_impl.cc:82] Allocation of 23721984 exceeds 10% of free system memory.
W0000 00:00:1779251519.543761   11731 cpu_allocator_impl.cc:82] Allocation of 23721984 exceeds 10% of free system memory.
W0000 00:00:1779251519.551303   11738 cpu_allocator_impl.cc:82] Allocation of 27998208 exceeds 10% of free system memory.


75/75 ━━━━━━━━━━━━━━━━━━━━ 7s 63ms/step - accuracy: 0.4792 - loss: 1.0012 - val_accuracy: 0.5717 - val_loss: 0.9014
Epoch 2/30
75/75 ━━━━━━━━━━━━━━━━━━━━ 4s 49ms/step - accuracy: 0.5942 - loss: 0.8543 - val_accuracy: 0.6533 - val_loss: 0.7546
Epoch 3/30
75/75 ━━━━━━━━━━━━━━━━━━━━ 4s 48ms/step - accuracy: 0.6413 - loss: 0.7675 - val_accuracy: 0.6833 - val_loss: 0.7110
Epoch 4/30
75/75 ━━━━━━━━━━━━━━━━━━━━ 4s 48ms/step - accuracy: 0.6754 - loss: 0.7141 - val_accuracy: 0.7200 - val_loss: 0.6571
Epoch 5/30
75/75 ━━━━━━━━━━━━━━━━━━━━ 4s 48ms/step - accuracy: 0.7292 - loss: 0.6253 - val_accuracy: 0.7600 - val_loss: 0.5931
Epoch 6/30
75/75 ━━━━━━━━━━━━━━━━━━━━ 4s 49ms/step - accuracy: 0.7471 - loss: 0.5974 - val_accuracy: 0.7383 - val_loss: 0.6255
Epoch 7/30
75/75 ━━━━━━━━━━━━━━━━━━━━ 4s 48ms/step - accuracy: 0.7725 - loss: 0.5292 - val_accuracy: 0.7850 - val_loss: 0.5288
Epoch 8/30
75/75 ━━━━━━━━━━━━━━━━━━━━ 4s 47ms/step - accuracy: 0.7933 - loss: 0.4966 - val_accuracy: 0.7883 - val_loss: 0.

In [15]:
from tensorflow import keras
import numpy as np
from PIL import Image

In [16]:
model=keras.models.load_model('./dsx06_2.keras')

In [17]:
mytestimages=['./cifar10/train/airplane/29.png','./cifar10/train/ship/8.png','./cifar10/train/truck/1.png',
              './cifar10/others/airplane/49994.png','./cifar10/others/ship/49985.png','./cifar10/others/truck/49997.png']
n=len(mytestimages)
test=np.zeros((n,32,32,3))
for i in range(n):
  img=Image.open(mytestimages[i])
  test[i]=np.array(img)

In [18]:
results = model.predict(test)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 112ms/step


In [19]:
np.set_printoptions(precision=6, floatmode='fixed', suppress=True)
print(results)

[[0.964548 0.035434 0.000018]
 [0.082127 0.917729 0.000144]
 [0.001889 0.000470 0.997642]
 [0.875345 0.092493 0.032162]
 [0.469589 0.492316 0.038095]
 [0.479795 0.099079 0.421126]]
